# 🎓 Lecture Transcriber

Reads your **GoodNotes recordings straight from GoodNotes' automatic backup in Google Drive**, transcribes them on Google's GPU, sorts them into your courses using your timetable, and saves **one readable transcript per lecture** to `Vorlesungen/Transkripte` in your Drive.

### Once
1. **iPad → GoodNotes → Settings → Cloud & Backup → Auto-backup:** on, **Google Drive**, file format **Goodnotes** (not PDF, or the recordings are left out).
2. **Google Drive:** create a folder **`Vorlesungen`** and put your TraiNex timetable (`.ics`) in it.
3. Optional: name each GoodNotes notebook after its module, e.g. *“M13 Multivariate Verfahren”*. Then recordings are filed correctly even when the timetable doesn't decide it.

### Every time
**Runtime → Run all** (Laufzeit → Alle ausführen), and allow Google Drive access. New recordings are transcribed; ones already done are skipped.

Anything else (laptop recordings via the app's *Send to Colab…*, other audio files, manual GoodNotes exports) goes into `Vorlesungen/Inbox`.

In [ ]:
#@title ⚙️ Settings
DRIVE_FOLDER = "Vorlesungen"  #@param {type:"string"}
PREVIEW_ONLY = False  #@param {type:"boolean"}
#@markdown ↑ tick to only *list* the recordings found and where they'd be filed (nothing is transcribed) — good for a first check
READ_GOODNOTES_BACKUP = True  #@param {type:"boolean"}
ONLY_THIS_SEMESTER = True  #@param {type:"boolean"}
#@markdown ↑ GoodNotes backs up *all* notebooks — this skips recordings from before the semester (judged by your timetable)
GOODNOTES_FOLDER = ""  #@param {type:"string"}
#@markdown ↑ optional: only notebooks in this GoodNotes folder, e.g. `Master` (empty = all)
#@markdown ---
MODEL = "large-v3"  #@param ["large-v3", "large-v3-turbo"]
LANGUAGE = "de"  #@param ["de", "en", "auto"]
#@markdown `large-v3` = most accurate (best on distant, noisy speech) · `large-v3-turbo` = ~2× faster, a little less accurate
#@markdown ---
#@markdown **Notion** (optional): put your integration token into 🔑 *Secrets* as `NOTION_TOKEN`, share the Notion page with the integration, and paste the page link here.
SEND_TO_NOTION = False  #@param {type:"boolean"}
NOTION_PAGE_LINK = ""  #@param {type:"string"}
#@markdown ---
DOWNLOAD_FOR_LAPTOP_APP = False  #@param {type:"boolean"}
#@markdown ↑ also download a zip for *Lecture library → Import from Colab…* on the laptop

In [ ]:
#@title 1 · Setup (GPU, code, Google Drive)
import os, subprocess, sys

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("❌ No GPU. Menu: Runtime → Change runtime type → T4 GPU → Save, then Run all again.")
print("✓ GPU:", gpu.stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "faster-whisper", "pypdf", "wordfreq"], check=True)
if os.path.isdir("/content/whisper"):
    subprocess.run(["git", "-C", "/content/whisper", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/RKIein/whisper", "/content/whisper"], check=True)
sys.path.insert(0, "/content/whisper")
import importlib
for name in ("course_calendar", "lecture_store", "transcriber", "notion_export", "colab_batch"):
    if name in sys.modules:                 # code updated since this session started → load the new version
        importlib.reload(sys.modules[name])
import colab_batch
print("✓ Code ready")

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"
BASE = f"{DRIVE}/{DRIVE_FOLDER}"
os.makedirs(f"{BASE}/Inbox", exist_ok=True)
BACKUP = colab_batch.find_goodnotes_backup(DRIVE) if READ_GOODNOTES_BACKUP else []
notebooks = colab_batch.backup_files(BACKUP, GOODNOTES_FOLDER)
inbox = colab_batch.inbox_files(f"{BASE}/Inbox")
ics = [f for f in os.listdir(BASE) if f.lower().endswith(".ics")]
if READ_GOODNOTES_BACKUP:
    print(f"✓ GoodNotes backup: {len(notebooks)} notebook(s) in {', '.join(os.path.basename(b) for b in BACKUP)}"
          if BACKUP else "⚠ No GoodNotes backup folder in your Drive yet — see “Once” above.")
print(f"✓ Inbox: {len(inbox)} file(s)")
print(f"✓ Timetable: {', '.join(ics)}" if ics else
      f"⚠ No timetable (.ics) in {DRIVE_FOLDER} — sorting only by notebook names.")

In [ ]:
#@title 2 · Load the speech model
if PREVIEW_ONLY:
    engine = None
    print("Preview: no model needed")
else:
    # installs missing CUDA libraries itself and test-runs the GPU once
    engine = colab_batch.WhisperEngine(MODEL, device="cuda")
    print(f"✓ {MODEL} loaded on the GPU")

In [ ]:
#@title 3 · Transcribe everything new
import json
publish = None
if SEND_TO_NOTION:
    from google.colab import userdata
    import notion_export as nx
    notion = nx.Notion(userdata.get("NOTION_TOKEN"))
    state = f"{BASE}/.state/notion.json"
    known = json.load(open(state)) if os.path.exists(state) else {}
    page = nx.page_id_from_url(NOTION_PAGE_LINK)
    if not page:
        raise SystemExit("❌ Paste the link of your Notion page into NOTION_PAGE_LINK (Settings).")
    db = nx.ensure_database(notion, page, known.get("database"))
    os.makedirs(f"{BASE}/.state", exist_ok=True)
    json.dump({"database": db}, open(state, "w"))
    publish = lambda groups: print(f"✓ Notion: {nx.publish(groups, notion, db, colab_batch.lecture_text)} new page(s)")

result = colab_batch.run(BASE, engine, language=None if LANGUAGE == "auto" else LANGUAGE,
                         backup_folders=BACKUP, publish=publish,
                         since="semester" if ONLY_THIS_SEMESTER else None, only_folder=GOODNOTES_FOLDER,
                         preview=PREVIEW_ONLY)

In [ ]:
#@title 4 · (optional) Download for the laptop app
from google.colab import files
if result.get("preview"):
    print(f"Preview saved in Drive → {DRIVE_FOLDER}/Results/{os.path.basename(result['preview'])}")
elif DOWNLOAD_FOR_LAPTOP_APP and result["zip"]:
    files.download(result["zip"])
elif result["transcripts"]:
    print(f"Your transcripts are in Google Drive → {DRIVE_FOLDER}/Transkripte")